# MaaS Platform and TNC Simulation
This notebook demonstrates how to run the simulation and analyze the outputs. First, we need to ensure the system path includes the `0-Simulation` folder so we can import the classes.

In [1]:
import sys
import os

# Add the 0-Simulation directory to Python path
simulation_path = os.path.join(os.getcwd(), '0-Simulation')
if simulation_path not in sys.path:
    sys.path.append(simulation_path)

from main import run_simulation

## Run Simulation with Different TNC Capacities
Execute the simulation using predefined TNC capacities. Results will be saved in the `2-Results` folder.

In [2]:
# Run simulations with different pooled capacities split across two TNC operators
capacities = [800]
number_days_list = [20000]

for i, cap in enumerate(capacities):
    print(f"\n{'='*60}")
    print(f"Running simulation with pooled TNC capacity = {cap}")
    print(f"{'='*60}")

    # Example split: 60/40 across TNC1/TNC2
    tnc_caps = {
        "TNC1": 0.5* cap,
        "TNC2": 0.5* cap,
    }

    run_simulation(
        tnc_capacities=tnc_caps,
        output_dir=f"./2-Results/capacity_{cap}",
        number_days=number_days_list[i],
        debug_enabled=True,
        enable_gradient_checks=False,
        logit_scale_mu = 0.7,
    )


Running simulation with pooled TNC capacity = 800


Simulation Progress:   0%|          | 0/20000 [00:00<?, ?day/s]

Lower level converged at Day 51 (stable for 50 iterations)
[Update 1] (step x1.0000) TNC1: fare=1.9434, cap_ratio=0.7996, lambda=0.000000
[Update 1] (step x1.0000) TNC2: fare=1.9434, cap_ratio=0.7996, lambda=0.000000
[Update 1] MaaS: fare=1.8659, share_TNC_per_type=[0.6000, 0.6141, 0.6025, 0.6027], lambda=0.000000
Lower level converged at Day 101 (stable for 50 iterations)
[Update 2] (step x1.0000) TNC1: fare=1.8999, cap_ratio=0.7993, lambda=0.000000
[Update 2] (step x1.0000) TNC2: fare=1.8999, cap_ratio=0.7993, lambda=0.000000
[Update 2] MaaS: fare=1.7750, share_TNC_per_type=[0.6000, 0.6257, 0.6043, 0.6050], lambda=0.000000
Lower level converged at Day 151 (stable for 50 iterations)
[Update 3] (step x1.0000) TNC1: fare=1.8649, cap_ratio=0.7991, lambda=0.000000
[Update 3] (step x1.0000) TNC2: fare=1.8649, cap_ratio=0.7991, lambda=0.000000
[Update 3] MaaS: fare=1.7519, share_TNC_per_type=[0.6000, 0.6297, 0.6060, 0.6072], lambda=0.000000
Lower level converged at Day 201 (stable for 50 it

## Load and Analyze Results
Parse the JSON outputs and visualizations from the simulation runs.

In [3]:
import json
import pandas as pd

# Load results from all simulations
results_summary = []

for cap in capacities:
    result_dir = f"./2-Results/v3_capacity_{cap}"
    result_file = os.path.join(result_dir, "final_results.json")

    if os.path.exists(result_file):
        with open(result_file, "r", encoding="utf-8") as f:
            results = json.load(f)

        maas_params = results.get("maas_params", {})
        share_vec = maas_params.get("share_TNC_per_traveler_type", [])

        tnc_profits = results.get("profits", {}).get("tnc", {})
        tnc_params = results.get("tnc_params", {})

        row = {
            "Pooled Capacity": cap,
            "MaaS Profit": results.get("profits", {}).get("maas"),
            "MaaS Fare": maas_params.get("fare"),
            "MaaS Share TNC Mean": float(sum(share_vec) / len(share_vec)) if share_vec else None,
            "MaaS Share TNC Vector": share_vec,
            "Pooled MaaS TNC Capacity vkm": maas_params.get("pooled_tnc_capacity_vkm"),
        }

        for name, profit in tnc_profits.items():
            row[f"{name} Profit"] = profit

        for name, params in tnc_params.items():
            row[f"{name} Fare"] = params.get("fare")
            row[f"{name} Capacity Ratio"] = params.get("capacity_ratio_to_MaaS")

        results_summary.append(row)

# Display results as a table
results_df = pd.DataFrame(results_summary)
print("\nSimulation Results Summary:")
print(results_df.to_string(index=False))


Simulation Results Summary:
Empty DataFrame
Columns: []
Index: []
